In [2]:
import os
import pandas as pd
import requests
from datetime import datetime

In [4]:
RAW_DATA_DIR = "../data/raw"

os.makedirs(RAW_DATA_DIR, exist_ok=True)

In [5]:
USGS_SITE = "01646500"
START_DATE = "2022-01-01"
END_DATE = "2024-01-01"

url = (
    "https://waterservices.usgs.gov/nwis/dv/"
    f"?format=json&sites={USGS_SITE}"
    "&parameterCd=00060"
    f"&startDT={START_DATE}&endDT={END_DATE}"
)

response = requests.get(url)
response.raise_for_status()

data = response.json()

In [7]:
records = data["value"]["timeSeries"][0]["values"][0]["value"]

usgs_df = pd.DataFrame(records)
usgs_df["dateTime"] = pd.to_datetime(usgs_df["dateTime"])
usgs_df["value"] = pd.to_numeric(usgs_df["value"], errors="coerce")

usgs_df = usgs_df[["dateTime", "value"]]
usgs_df.columns = ["timestamp", "streamflow_cfs"]

usgs_df.head()


,timestamp,streamflow_cfs
0,2022-01-01,3370
1,2022-01-02,5320
2,2022-01-03,7380
3,2022-01-04,13800
4,2022-01-05,13600


In [8]:
usgs_path = os.path.join(RAW_DATA_DIR, "usgs_streamflow.csv")
usgs_df.to_csv(usgs_path, index=False)

usgs_path


'../data/raw/usgs_streamflow.csv'

In [9]:
rain_dates = pd.date_range(start=START_DATE, end=END_DATE, freq="D")

rain_df = pd.DataFrame({
    "timestamp": rain_dates,
    "rainfall_in": pd.Series(
        pd.np.random.gamma(shape=0.6, scale=0.4, size=len(rain_dates))
    )
})

rain_df["rainfall_in"] = rain_df["rainfall_in"].round(2)
rain_df.head()


/var/folders/rc/c2ysvgjd75xdxm3zx8fz_t640000gn/T/ipykernel_89585/801084642.py:6: FutureWarning: The pandas.np module is deprecated and will be removed from pandas in a future version. Import numpy directly instead.
  pd.np.random.gamma(shape=0.6, scale=0.4, size=len(rain_dates))


,timestamp,rainfall_in
0,2022-01-01,0.04
1,2022-01-02,0.03
2,2022-01-03,0.47
3,2022-01-04,0.05
4,2022-01-05,0.85


In [10]:
rain_path = os.path.join(RAW_DATA_DIR, "rainfall_daily.csv")
rain_df.to_csv(rain_path, index=False)

rain_path


'../data/raw/rainfall_daily.csv'

In [11]:
flow_df = usgs_df.copy()

flow_df["interceptor_flow_gpm"] = (
    flow_df["streamflow_cfs"] * 448.831 +
    pd.np.random.normal(0, 5000, size=len(flow_df))
)

flow_df["interceptor_flow_gpm"] = flow_df["interceptor_flow_gpm"].clip(lower=0)
flow_df = flow_df[["timestamp", "interceptor_flow_gpm"]]

flow_df.head()


/var/folders/rc/c2ysvgjd75xdxm3zx8fz_t640000gn/T/ipykernel_89585/3052062017.py:5: FutureWarning: The pandas.np module is deprecated and will be removed from pandas in a future version. Import numpy directly instead.
  pd.np.random.normal(0, 5000, size=len(flow_df))


,timestamp,interceptor_flow_gpm
0,2022-01-01,1.509371e+06
1,2022-01-02,2.388439e+06
2,2022-01-03,3.306823e+06
3,2022-01-04,6.198517e+06
4,2022-01-05,6.099982e+06


In [12]:
flow_path = os.path.join(RAW_DATA_DIR, "interceptor_flow.csv")
flow_df.to_csv(flow_path, index=False)

flow_path


'../data/raw/interceptor_flow.csv'